# AutoGluon

In [1]:
import qlib
import pandas as pd
import numpy as np
from qlib.contrib.data.handler import Alpha158
from qlib.data.dataset import DatasetH
from autogluon.tabular import TabularPredictor

In [2]:
# 1. Inicializar Qlib
qlib.init(provider_uri='/home/toni/.qlib/qlib_data/us_data',region=qlib.constant.REG_US)

[10681:MainThread](2026-05-06 13:11:22,598) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[10681:MainThread](2026-05-06 13:11:23,484) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[10681:MainThread](2026-05-06 13:11:23,485) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/toni/.qlib/qlib_data/us_data')}


In [3]:
import qlib
from qlib.constant import REG_US
from qlib.data import D

qlib.init(provider_uri="/home/toni/.qlib/qlib_data/us_data", region=REG_US)

df = D.features(["^GSPC"], ["$close", "$factor"], start_time="2020-01-01", end_time="2026-05-01", freq="day")

print(df.shape)
print(df.empty)
print(df.tail())


[10681:MainThread](2026-05-06 13:11:23,496) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[10681:MainThread](2026-05-06 13:11:23,498) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[10681:MainThread](2026-05-06 13:11:23,499) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/toni/.qlib/qlib_data/us_data')}


(1591, 2)
False
                         $close   $factor
instrument datetime                      
^GSPC      2026-04-27  4.882702  0.000681
           2026-04-28  4.858805  0.000681
           2026-04-29  4.856866  0.000681
           2026-04-30  4.906591  0.000681
           2026-05-01  4.920960  0.000681


In [4]:
# 2. Cargar dataset Alpha154/158
data_handler_config= {
    "start_time":"2018-01-01",
    "end_time":"2026-04-01",
    "fit_start_time":"2018-01-01",
    "fit_end_time":"2024-12-31",
    "instruments":"sp500",
}

handler= Alpha158(**data_handler_config)

[10681:MainThread](2026-05-06 13:11:34,081) INFO - qlib.timer - [log.py:127] - Time cost: 10.501s | Loading data Done
[10681:MainThread](2026-05-06 13:11:34,724) INFO - qlib.timer - [log.py:127] - Time cost: 0.202s | DropnaLabel Done
[10681:MainThread](2026-05-06 13:11:35,979) INFO - qlib.timer - [log.py:127] - Time cost: 1.254s | CSZScoreNorm Done
[10681:MainThread](2026-05-06 13:11:35,982) INFO - qlib.timer - [log.py:127] - Time cost: 1.899s | fit & process data Done
[10681:MainThread](2026-05-06 13:11:35,982) INFO - qlib.timer - [log.py:127] - Time cost: 12.403s | Init data Done


In [5]:
# 3. Obtener features y labels
features_df= handler.fetch(col_set="feature")
labels_df= handler.fetch(col_set="label")

In [6]:
label_column='LABEL0'

In [7]:
# 4. Combinar en un solo DataFrame
data= pd.concat([features_df, labels_df],axis=1)
# Remove columns that are completely unavailable, e.g. VWAP0 if no $vwap exists
data = data.dropna(axis=1, how="all")

# Make sure label is numeric and finite
data[label_column] = pd.to_numeric(data[label_column], errors="coerce")
data = data.replace([np.inf, -np.inf], np.nan)
data = data[np.isfinite(data[label_column])]

print("Rows after label cleanup:", len(data))
print("Remaining label NaNs:", data[label_column].isna().sum())

Rows after label cleanup: 1018226
Remaining label NaNs: 0


In [8]:
data

KMID      KLEN     KMID2       KUP      KUP2  \
datetime   instrument                                                     
2018-01-02 A           0.002670  0.008158  0.327272  0.004301  0.527267   
           AAL         0.012612  0.022931  0.550002  0.002102  0.091663   
           AAP         0.051437  0.081467  0.631385  0.018236  0.223845   
           AAPL        0.012341  0.017866  0.690783  0.000235  0.013159   
           ABBV        0.013074  0.022133  0.590698  0.005044  0.227907   
...                         ...       ...       ...       ...       ...   
2026-04-01 XYZ        -0.026647  0.032287 -0.825313  0.004169  0.129117   
           YUM        -0.022203  0.029137 -0.762011  0.000000  0.000000   
           ZBH         0.005634  0.010826  0.520410  0.001436  0.132655   
           ZBRA       -0.007802  0.026183 -0.297989  0.006127  0.234003   
           ZTS        -0.007699  0.018020 -0.427229  0.005838  0.323948   

                           KLOW     KLOW2      KSFT     KSFT2     OPEN0  ...  \
datetime   instrument                                                    ...   
2018-01-02 A           0.001187  0.145461 -0.000445 -0.054535  0.997337  ...   
           AAL         0.008217  0.358334  0.018727  0.816673  0.987545  ...   
           AAP         0.011794  0.144769  0.044995  0.552309  0.951079  ...   
           AAPL        0.005289  0.296058  0.017395  0.973681  0.987809  ...   
           ABBV        0.004015  0.181395  0.012044  0.544186  0.987095  ...   
...                         ...       ...       ...       ...       ...  ...   
2026-04-01 XYZ         0.001471  0.045570 -0.029344 -0.908861  1.027376  ...   
           YUM         0.006934  0.237989 -0.015268 -0.524022  1.022707  ...   
           ZBH         0.003756  0.346934  0.007954  0.734689  0.994397  ...   
           ZBRA        0.012254  0.468008 -0.001675 -0.063983  1.007864  ...   
           ZTS         0.004484  0.248823 -0.009053 -0.502353  1.007758  ...   

                        VSUMN10   VSUMN20   VSUMN30   VSUMN60    VSUMD5  \
datetime   instrument                                                     
2018-01-02 A           0.726731  0.586454  0.532266  0.515107 -0.083773   
           AAL         0.634902  0.533584  0.504173  0.500709  0.456964   
           AAP         0.403379  0.482207  0.490406  0.494112  0.388357   
           AAPL        0.608529  0.550233  0.495184  0.494970  0.211619   
           ABBV        0.737471  0.504350  0.497144  0.499255  0.472005   
...                         ...       ...       ...       ...       ...   
2026-04-01 XYZ         0.620792  0.567859  0.512933  0.507910  0.166319   
           YUM         0.527585  0.510032  0.511144  0.520642 -0.029544   
           ZBH         0.462439  0.453934  0.497771  0.492654  0.415168   
           ZBRA        0.485893  0.516755  0.516473  0.504114  0.435889   
           ZTS         0.574554  0.564000  0.531321  0.518275 -0.186424   

                        VSUMD10   VSUMD20   VSUMD30   VSUMD60    LABEL0  
datetime   instrument                                                    
2018-01-02 A          -0.453462 -0.172908 -0.064533 -0.030214 -0.007501  
           AAL        -0.269803 -0.067167 -0.008345 -0.001418  0.006305  
           AAP         0.193242  0.035586  0.019188  0.011776  0.036898  
           AAPL       -0.217057 -0.100466  0.009631  0.010061  0.004645  
           ABBV       -0.474942 -0.008701  0.005711  0.001491 -0.005702  
...                         ...       ...       ...       ...       ...  
2026-04-01 XYZ        -0.241585 -0.135718 -0.025867 -0.015820  0.015055  
           YUM        -0.055170 -0.020064 -0.022288 -0.041285  0.008136  
           ZBH         0.075121  0.092132  0.004458  0.014693  0.001210  
           ZBRA        0.028214 -0.033509 -0.032946 -0.008228  0.040820  
           ZTS        -0.149107 -0.127999 -0.062642 -0.036550  0.002713  

[1018226 rows x 158 columns]

In [9]:
# 5. Preparar train/test split temporal (importante en finanzas!)
dt = data.index.get_level_values("datetime")
train_data = data[dt < "2023-01-01"]
test_data = data[dt >= "2023-01-01"]

train_data = train_data.reset_index()
test_data = test_data.reset_index()

print(f"Train samples:{len(train_data)}, Test samples:{len(test_data)}")
print(f"Features:{features_df.shape[1]}")

Train samples:614648, Test samples:403578
Features:158


In [10]:
# 6. Entrenar con AutoGluon
label_column='LABEL0'# Nombre típico de la columna objetivo en Qlib

predictor= TabularPredictor(
    label=label_column,
    path='qlib_autogluon_models/',
    eval_metric='rmse'# o 'mae' para regresión
).fit(
    train_data=train_data,
    time_limit=3600,# 1 hora
    presets='best_quality',
    verbosity=2
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Tue Nov 5 00:21:55 UTC 2024
CPU Count:          22
Pytorch Version:    2.7.0
CUDA Version:       12.8
GPU Memory:         GPU 0: 15.99/15.99 GB
Total GPU Memory:   Free: 15.99 GB, Allocated: 0.00 GB, Total: 15.99 GB
GPU Count:          1
Memory Avail:       8.05 GB / 15.46 GB (52.1%)
Disk Space Avail:   441.90 GB / 1896.54 GB (23.3%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked over

In [11]:
# 7. Evaluar
test_features= test_data.drop(columns=[label_column])
test_labels= test_data[label_column]

predictions= predictor.predict(test_features)
performance= predictor.evaluate(test_data)

print(f"\n📈 Performance en Test:")
print(performance)

# 8. Leaderboard de modelos
predictor.leaderboard(test_data)


📈 Performance en Test:
{'root_mean_squared_error': np.float32(-0.020126244), 'mean_squared_error': -0.00040506571531295776, 'mean_absolute_error': -0.013455897569656372, 'r2': -0.028998255729675293, 'pearsonr': 0.018238509073853493, 'median_absolute_error': -0.009555637836456299}


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBMXT_BAG_L1,-0.019993,-0.021297,root_mean_squared_error,3.273487,0.657307,1606.495204,3.273487,0.657307,1606.495204,1,True,1
1,WeightedEnsemble_L2,-0.019993,-0.021297,root_mean_squared_error,3.283858,0.663223,1606.506069,0.010371,0.005916,0.010865,2,True,2
2,WeightedEnsemble_L3,-0.020126,-0.021211,root_mean_squared_error,4.807874,1.001060,2220.265886,0.012673,0.004141,0.138149,3,True,4
3,LightGBMXT_BAG_L2,-0.020281,-0.021269,root_mean_squared_error,4.795201,0.996919,2220.127736,1.521714,0.339612,613.632532,2,True,3
